Question
20
Weeky Assessment -Agentic Ai and RAG
Description
AI Internship Practical Assessment
⏱️ Duration: 60 Minutes
📊 Total Marks: 50

🔷 Question 1: Build an End-to-End RAG + Agent System (25 Marks)
🧩 Scenario
You are an AI intern at an ed-tech company. Students frequently ask questions about:

Course policies (refunds, deadlines)
Lecture content (PDF notes)
Assignment deadlines
Their enrollment status
The company wants a single intelligent assistant that:

Answers questions using internal documents (PDFs, FAQs)
Fetches student-specific data (like enrollment or progress) using tools/APIs
Avoids hallucination and gives reliable answers
💻 Task
Design and implement a working prototype (pseudo-code or real code) for this system.

Your solution must include:

✅ 1. RAG Pipeline
How you will ingest and preprocess documents
Chunking strategy (with justification)
Embedding + vector store choice
Retrieval logic
How context is injected into the LLM
✅ 2. Agent Integration
Design an agent that decides:
When to use RAG
When to call a tool (e.g., get_student_status(student_id))
Show how tools are defined and used
✅ 3. End-to-End Flow
Write code or structured pseudo-code showing:

Input query
Retrieval step
Tool calling (if needed)
Final answer generation
✅ 4. Reliability Improvements
Show at least 2 techniques in code/design to:

Reduce hallucination
Improve answer quality
🎯 Expected Outcome
A working architecture/code that demonstrates:

RAG + Agent working together
Decision-making capability
Real-world practicality
🔷 Question 2: Design a Multi-Agent Workflow with LangGraph (25 Marks)
🧩 Scenario
You are building an AI-powered customer support system for a fintech company.

The system must handle:

Transaction queries
Fraud detection flags
Refund requests
General FAQs
The company wants:

High accuracy
Step-by-step reasoning
Ability to retry if answer is incorrect
Modular, scalable architecture
💻 Task
Design and implement a multi-agent workflow using LangGraph (or similar framework).

✅ 1. Agent Design
Define at least 3 agents, such as:

Retrieval Agent
Reasoning/Answer Agent
Validation Agent
Explain briefly (in comments or code):

Each agent’s role
Input/output
✅ 2. Graph Workflow Implementation
Write code or pseudo-code to:

Define state
Add nodes (agents)
Define edges
Implement conditional logic
👉 Must include:

Retry loop if validation fails
Clear start and end states
✅ 3. State Management
Show how state evolves across steps:

Query
Context
Intermediate reasoning
Final answer
Validation flag
✅ 4. Task Delegation & Communication
Demonstrate:

How agents pass information
How decisions are made between agents
🎯 Expected Outcome
A clear multi-step, graph-based agent system that:

Handles complex queries
Demonstrates reasoning + validation
Uses proper orchestration

In [2]:
# ==========================================================
# FULL RAG + AGENT SYSTEM (ERROR-FREE VERSION)
# ==========================================================

# -------------------------------
# STEP 0: INSTALL LIBRARIES
# -------------------------------
try:
    from pypdf import PdfReader
except:
    print("Installing missing libraries...")
    !pip install pypdf chromadb sentence-transformers transformers torch
    from pypdf import PdfReader

from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline

# -------------------------------
# STEP 1: LOAD PDF (SAFE)
# -------------------------------

def load_pdf(path):
    try:
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text()
        return text
    except:
        print("⚠️ PDF not found. Using dummy content...")
        return """
        Refund policy: Students can request refund within 7 days.
        Assignments must be submitted before deadline.
        Course access valid for 6 months.
        """

# -------------------------------
# STEP 2: CHUNKING
# -------------------------------

def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+chunk_size])
        i += chunk_size - overlap
    return chunks

# -------------------------------
# STEP 3: EMBEDDINGS + DB
# -------------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()

try:
    client.delete_collection("edtech_docs")
except:
    pass

collection = client.create_collection("edtech_docs")

def store_chunks(chunks):
    for i, chunk in enumerate(chunks):
        emb = embedding_model.encode(chunk).tolist()
        collection.add(
            documents=[chunk],
            embeddings=[emb],
            ids=[str(i)]
        )

# -------------------------------
# STEP 4: RETRIEVAL
# -------------------------------

def retrieve(query, k=3):
    q_emb = embedding_model.encode(query).tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=k)
    return results["documents"][0]

# -------------------------------
# STEP 5: TOOL (API)
# -------------------------------

def get_student_status(student_id):
    return f"Student {student_id} enrolled in AI course, progress 75%, next deadline in 5 days"

# -------------------------------
# STEP 6: LLM
# -------------------------------

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

# -------------------------------
# STEP 7: AGENT ROUTER
# -------------------------------

def agent_router(query):
    if "my" in query or "status" in query or "enrollment" in query:
        return "tool"
    return "rag"

# -------------------------------
# STEP 8: PROMPT BUILDER
# -------------------------------

def build_prompt(context, query):
    return f"""
You are an ed-tech assistant.

Answer ONLY using the context below.
If not found, say "Not found".

Context:
{context}

Question:
{query}

Answer:
"""

# -------------------------------
# STEP 9: AGENTS
# -------------------------------

def retrieval_agent(state):
    state["context"] = retrieve(state["query"], k=5)
    return state

def reasoning_agent(state):
    context = " ".join(state["context"])
    prompt = build_prompt(context, state["query"])
    state["answer"] = llm(prompt)[0]["generated_text"]
    return state

def validation_agent(state):
    if "Not found" in state["answer"]:
        state["valid"] = False
    else:
        state["valid"] = True
    return state

# -------------------------------
# STEP 10: MAIN SYSTEM
# -------------------------------

def system(query, student_id=None):

    print("\nUser Query:", query)

    decision = agent_router(query)

    # TOOL PATH
    if decision == "tool":
        print("[Agent] Using TOOL")

        tool_data = get_student_status(student_id)

        prompt = f"""
Answer using the data:

{tool_data}

Question: {query}
Answer:
"""
        return llm(prompt)[0]["generated_text"]

    # RAG PATH
    else:
        print("[Agent] Using RAG")

        state = {
            "query": query,
            "context": [],
            "answer": "",
            "valid": False
        }

        state = retrieval_agent(state)
        state = reasoning_agent(state)
        state = validation_agent(state)

        # Retry once
        if not state["valid"]:
            print("[Validation] Retrying...")
            state = reasoning_agent(state)

        return state["answer"]

# -------------------------------
# STEP 11: RUN SYSTEM
# -------------------------------

# Load data
text = load_pdf("course_policy.pdf")
chunks = chunk_text(text)
store_chunks(chunks)

# Test queries
print("\n--- RAG QUERY ---")
print(system("What is refund policy?"))

print("\n--- TOOL QUERY ---")
print(system("What is my enrollment status?", student_id=101))

Installing missing libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.0 MB/s eta 0:00:00
  Attempting uninst

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

[Validation] Retrying...


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an ed-tech assistant.

Answer ONLY using the context below.
If not found, say "Not found".

Context:

        Refund policy: Students can request refund within 7 days.
        Assignments must be submitted before deadline.
        Course access valid for 6 months.
        

Question:
What is refund policy?

Answer:


--- TOOL QUERY ---

User Query: What is my enrollment status?
[Agent] Using TOOL


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer using the data:

Student 101 enrolled in AI course, progress 75%, next deadline in 5 days

Question: What is my enrollment status?
Answer:

